# Lecture 07 – GLUE & SuperGLUE Benchmark Datasets

This notebook explores the **GLUE** and **SuperGLUE** benchmarks and evaluates
five pre-trained language models in a zero-shot setting:

| Model | Type | Size | Architecture |
|-------|------|------|-------------|
| `gpt2` | Causal LM | 117M | Decoder-only |
| `bert-base-uncased` | Masked LM | 110M | Encoder-only |
| `bert-large-uncased` | Masked LM | 340M | Encoder-only |
| `Qwen/Qwen3-0.6B` | Instruction-tuned causal LM | 0.6B | Decoder-only |
| `Qwen/Qwen3-1.7B` | Instruction-tuned causal LM | 1.7B | Decoder-only |

**Zero-shot strategies per architecture:**
- **BERT (masked LM):** fill-mask scoring — compare token probabilities at `[MASK]`
- **GPT-2 (causal LM):** likelihood scoring — compare log P(label | prompt)
- **Qwen3 (instruct):** chat-format prompting — model returns label as text


---
## 1. Setup


In [3]:
import os, warnings
warnings.filterwarnings('ignore')

os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'   # mirror for China
# os.environ['HF_HOME'] = '/inspire/hdd/project/fdu-aidake-cfff/public/hf-home'

import sys
sys.path.insert(0, '../')
from utils.helper import setup_notebook
setup_notebook()

import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, get_dataset_config_names
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    AutoModelForMaskedLM, pipeline,
)

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'device: {device}')


device: mps


---
## 2. GLUE Dataset Exploration

- **Paper:** Wang et al., *GLUE: A Multi-Task Benchmark and Analysis Platform for Natural Language Understanding*, 2018  
- **HuggingFace:** [https://huggingface.co/datasets/nyu-mll/glue](https://huggingface.co/datasets/nyu-mll/glue)
- **URL:** [https://gluebenchmark.com/](https://gluebenchmark.com/)
- **General Language Understanding Evaluation (GLUE) benchmark**: A collection of NLU tasks including QA, sentiment analysis, and textual entailment, and an associated online platform for model evaluation, comparison, and analysis.

| Task | Category | Description | Metric |
|------|----------|-------------|--------|
| CoLA | Single-sentence | Grammatical acceptability | MCC |
| SST-2 | Single-sentence | Sentiment (positive / negative) | Accuracy |
| MRPC | Similarity | Semantic equivalence of sentence pairs | Accuracy, F1 |
| STS-B | Similarity | Similarity score [1, 5] | Pearson / Spearman |
| QQP | Similarity | Duplicate question detection | Accuracy, F1 |
| MNLI | Inference | 3-class NLI (entail / neutral / contradict) | Accuracy |
| QNLI | Inference | Does context contain answer? | Accuracy |
| RTE | Inference | 2-class NLI (entail / not entail) | Accuracy |
| WNLI | Inference | Winograd coreference → entailment | Accuracy |

<img src="figs/glue_tasks_table.png" alt="GLUE tasks table" width="80%">


In [4]:
configs = get_dataset_config_names('nyu-mll/glue')
print('GLUE tasks:', configs)

GLUE tasks: ['ax', 'cola', 'mnli', 'mnli_matched', 'mnli_mismatched', 'mrpc', 'qnli', 'qqp', 'rte', 'sst2', 'stsb', 'wnli']


### 2.1 Single-sentence task: CoLA — Corpus of Linguistic Acceptability

Binary label: `0` = unacceptable, `1` = acceptable.  
Metric: **Matthews Correlation Coefficient (MCC)** — accounts for class imbalance.

- Corpus of Linguistic Acceptability (CoLA): English acceptability judgments drawn from books and journal articles on linguistic theory
- Each example is a sequence of words annotated with whether it is a grammatical English sentence. Score is between [-1,1]


In [5]:
cola = load_dataset('nyu-mll/glue', 'cola')
print(cola)
df = pd.DataFrame(cola['train'][:8])
df

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 8551
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1043
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1063
    })
})


,sentence,label,idx
0,"Our friends won't buy this analysis, let alone...",1,0
1,One more pseudo generalization and I'm giving up.,1,1
2,One more pseudo generalization or I'm giving up.,1,2
3,"The more we study verbs, the crazier they get.",1,3
4,Day by day the facts are getting murkier.,1,4
5,I'll fix you a drink.,1,5
6,Fred watered the plants flat.,1,6
7,Bill coughed his way out of the restaurant.,1,7


### 2.2 Singel-sentence task: SST-2 — Stanford Sentiment Treebank

Binary label: `0` = negative, `1` = positive.  
Metric: **Accuracy**. 67K training sentences from movie reviews.

- Sentences from movie reviews and human annotations of their sentiment. The task is to predict the sentiment of a given sentence. We use the two-way (positive/negative) class split, and use only sentence-level labels.

In [6]:
sst2 = load_dataset('nyu-mll/glue', 'sst2')
print(sst2)
df = pd.DataFrame(sst2['train'][:8])
df

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})


,sentence,label,idx
0,hide new secretions from the parental units,0,0
1,"contains no wit , only labored gags",0,1
2,that loves its characters and communicates som...,1,2
3,remains utterly satisfied to remain the same t...,0,3
4,on the worst revenge-of-the-nerds clichés the ...,0,4
5,that 's far too tragic to merit such superfici...,0,5
6,demonstrates that the director of such hollywo...,1,6
7,of saucy,1,7


### 2.3 Similarity and Paraphrase Tasks: MRPC — Microsoft Research Paraphrase Corpus

Binary label: `0` = not paraphrase, `1` = paraphrase.  
Metric: **Accuracy + F1**. Class imbalanced: 68% positive pairs.

- Sentence pairs with human annotations whether sentences are semantically equivalent
- Classes are imbalanced (68% positive)


In [7]:
mrpc = load_dataset('nyu-mll/glue', 'mrpc')
print(mrpc)
df = pd.DataFrame(mrpc['train'][:5])
df

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})


,sentence1,sentence2,label,idx
0,"Amrozi accused his brother , whom he called "" ...","Referring to him as only "" the witness "" , Amr...",1,0
1,Yucaipa owned Dominick 's before selling the c...,Yucaipa bought Dominick 's in 1995 for $ 693 m...,0,1
2,They had published an advertisement on the Int...,"On June 10 , the ship 's owners had published ...",1,2
3,"Around 0335 GMT , Tab shares were up 19 cents ...","Tab shares jumped 20 cents , or 4.6 % , to set...",0,3
4,"The stock rose $ 2.11 , or about 11 percent , ...",PG & E Corp. shares jumped $ 1.63 or 8 percent...,1,4


### 2.4 Similarity and Paraphrase Tasks: STS-B — Semantic Textual Similarity Benchmark

Label: continuous score in **[1.0, 5.0]** (human-annotated similarity).  
Metric: **Pearson** and **Spearman** correlation. This is the only regression task in GLUE.

- Sentence pairs drawn from various sources, human annotated with similarity score from 1 to 5
- Task: predict [1,5]. Report Pearson and Spearman correlation coefficients

In [8]:
stsb = load_dataset('nyu-mll/glue', 'stsb')
print(stsb)
df = pd.DataFrame(stsb['train'][:5])
df

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 5749
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1379
    })
})


,sentence1,sentence2,label,idx
0,A plane is taking off.,An air plane is taking off.,5.00,0
1,A man is playing a large flute.,A man is playing a flute.,3.80,1
2,A man is spreading shreded cheese on a pizza.,A man is spreading shredded cheese on an uncoo...,3.80,2
3,Three men are playing chess.,Two men are playing chess.,2.60,3
4,A man is playing the cello.,A man seated is playing the cello.,4.25,4


### 2.5 Similarity and Paraphrase Tasks: QQP — Quora Question Pairs

Binary label: `0` = not duplicate, `1` = duplicate.  
Metric: **Accuracy + F1**. 364K training pairs; 63% negative (imbalanced).

- Question pairs from community QA website, Quora
- Classes are imbalanced (63% negative) so accuracy and F1 score is reported


In [9]:
qqp = load_dataset('nyu-mll/glue', 'qqp')
print(qqp)
df = pd.DataFrame(qqp['train'][:5])
df

DatasetDict({
    train: Dataset({
        features: ['question1', 'question2', 'label', 'idx'],
        num_rows: 363846
    })
    validation: Dataset({
        features: ['question1', 'question2', 'label', 'idx'],
        num_rows: 40430
    })
    test: Dataset({
        features: ['question1', 'question2', 'label', 'idx'],
        num_rows: 390965
    })
})


,question1,question2,label,idx
0,How is the life of a math student? Could you d...,Which level of prepration is enough for the ex...,0,0
1,How do I control my horny emotions?,How do you control your horniness?,1,1
2,What causes stool color to change to yellow?,What can cause stool to come out as little balls?,0,2
3,What can one do after MBBS?,What do i do after my MBBS ?,1,3
4,Where can I find a power outlet for my laptop ...,"Would a second airport in Sydney, Australia be...",0,4


### 2.6 Inference Tasks: MNLI — Multi-Genre Natural Language Inference

3-class label: `0` = entailment, `1` = neutral, `2` = contradiction.  
Metric: **Accuracy** on matched + mismatched genres. 393K training pairs.

- Crowd-sourced sentence pairs with entailment annotations: entailment (0), contradiction (2) and neutral (1)
- Task: predict these three labels


In [10]:
mnli = load_dataset('nyu-mll/glue', 'mnli')
print(mnli)
df = pd.DataFrame(mnli['train'][:6])
df

DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 392702
    })
    validation_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9815
    })
    validation_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9832
    })
    test_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9796
    })
    test_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9847
    })
})


,premise,hypothesis,label,idx
0,Conceptually cream skimming has two basic dime...,Product and geography are what make cream skim...,1,0
1,you know during the season and i guess at at y...,You lose the things to the following level if ...,0,1
2,One of our number will carry out your instruct...,A member of my team will execute your orders w...,0,2
3,How do you know? All this is their information...,This information belongs to them.,0,3
4,yeah i tell you what though if you go price so...,The tennis shoes have a range of prices.,1,4
5,my walkman broke so i'm upset now i just have ...,I'm upset that my walkman broke and now I have...,0,5


### 2.7 Inference Tasks: QNLI, RTE, WNLI

| Task | Label | Train size | Notes |
|------|-------|-----------|-------|
| QNLI | 0=entailment / 1=not | 105K | SQuAD → sentence pair classification |
| RTE | 0=entailment / 1=not | 2.5K | Combines RTE1/3/5 shared tasks |
| WNLI | 0=not / 1=entailment | 634 | Winograd coreference; adversarial dev set |

- The Stanford Question Answering Dataset (QNLI based on SQuAD)
  - Task converted into sentence pair classification by forming a pair between each question and each sentence in the corresponding context, and filtering out pairs with low lexical overlap between the question and the context sentence
  - Task: predict yes (0) or no (1), the context sentence contains the answer to the question
- The Recognizing Textual Entailment (RTE)
  - The task is to predict entailment (0) or not entailment (1) between pairs of sentences
- The Winograd Schema Challenge (WNLI)
  - Reading comprehension task to identify referent of a pronoun using entailment between two sentences (one has pronoun reference explicit)
  - Predict 1 (entailment) or 0 (not entailment)
  - Designed to fool simple statistical techniques
  - Test set is imbalanced (65% not entailment) and dev set is adversarial (memorization will hurt performance)


In [11]:
for task in ['qnli', 'rte', 'wnli']:
    ds = load_dataset('nyu-mll/glue', task)
    train = ds['train']
    print(f'{task.upper():6s}: train={len(train):,}  keys={list(train.features.keys())}')
    ex = train[0]
    for k, v in ex.items():
        print(f'       {k}: {str(v)[:80]}')
    print()

QNLI  : train=104,743  keys=['question', 'sentence', 'label', 'idx']
       question: When did the third Digimon series begin?
       sentence: Unlike the two seasons before it and most of the seasons that followed, Digimon 
       label: 1
       idx: 0

RTE   : train=2,490  keys=['sentence1', 'sentence2', 'label', 'idx']
       sentence1: No Weapons of Mass Destruction Found in Iraq Yet.
       sentence2: Weapons of Mass Destruction Found in Iraq.
       label: 1
       idx: 0

WNLI  : train=635  keys=['sentence1', 'sentence2', 'label', 'idx']
       sentence1: I stuck a pin through a carrot. When I pulled the pin out, it had a hole.
       sentence2: The carrot had a hole.
       label: 1
       idx: 0



---
## 3. Zero-shot Evaluation on GLUE

We evaluate on three representative tasks using 200 validation examples each:

| Task | Labels | Zero-shot strategy |
|------|--------|-----------------|
| SST-2 | negative / positive | Fill-mask / likelihood / chat |
| MRPC | not paraphrase / paraphrase | Fill-mask / likelihood / chat |
| RTE | entailment / not entailment | Fill-mask / likelihood / chat |
| MNLI | entailment / neutral / contradiction | Fill-mask / likelihood / chat |

**Evaluation budget:** 200 examples per task (validation split), to run in minutes on CPU.


### 3.1 Shared evaluation helpers


In [12]:
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef

def report(name, task, gold, pred):
    acc = accuracy_score(gold, pred)
    if task == 'cola':
        extra = f'  MCC={matthews_corrcoef(gold, pred):.3f}'
    elif task in ('mrpc', 'qqp'):
        extra = f'  F1={f1_score(gold, pred):.3f}'
    elif task == 'mnli':
        extra = f'  Macro-F1={f1_score(gold, pred, average="macro"):.3f}'
    else:
        extra = ''
    print(f'{name:25s}  acc={acc:.3f}{extra}')
    return acc

EVAL_N = 200   # number of validation examples to evaluate

### 3.2 SST-2 — Sentiment Classification

**Zero-shot templates:**
- BERT fill-mask: `"{sentence} It was [MASK]."` → compare P(`great`) vs P(`terrible`)
- GPT-2 likelihood: compare log P(`" Positive"`) vs log P(`" Negative"`) after the sentence
- Qwen3 chat: *Classify the sentiment of the following sentence as 'positive' or 'negative'.*


In [13]:
sst2_val = sst2['validation'].select(range(EVAL_N))
sst2_gold = sst2_val['label']
sst2_sentences = sst2_val['sentence']
print(f'SST-2 validation sample: {len(sst2_val)} examples')
print('Label distribution:', dict(zip(*np.unique(sst2_gold, return_counts=True))))

SST-2 validation sample: 200 examples
Label distribution: {np.int64(0): np.int64(101), np.int64(1): np.int64(99)}


#### BERT — fill-mask scoring


In [14]:
# Load BERT models (encoder-only)
print('Loading bert-base-uncased...')
bert_base_tok = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_base_mlm = AutoModelForMaskedLM.from_pretrained('bert-base-uncased').to(device).eval()

print('Loading bert-large-uncased...')
bert_large_tok = AutoTokenizer.from_pretrained('bert-large-uncased')
bert_large_mlm = AutoModelForMaskedLM.from_pretrained('bert-large-uncased').to(device).eval()
print('Done.')


Loading bert-base-uncased...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-large-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Done.


In [18]:
def bert_fillmask_sst2(tokenizer, model, sentences, pos_word='great', neg_word='terrible'):
    """Zero-shot SST-2 via fill-mask: compare P(pos_word) vs P(neg_word) at [MASK]."""
    pos_id = tokenizer.convert_tokens_to_ids(pos_word)
    neg_id = tokenizer.convert_tokens_to_ids(neg_word)
    preds = []
    for sent in sentences:
        text = f'{sent} It was [MASK].'
        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(device)
        mask_idx = (inputs['input_ids'][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]
        with torch.no_grad():
            logits = model(**inputs).logits[0, mask_idx, :]  # (1, vocab)
        p_pos = logits[0, pos_id].item()
        p_neg = logits[0, neg_id].item()
        preds.append(1 if p_pos > p_neg else 0)
    return preds

print('Evaluating bert-base-uncased on SST-2...')
pred_base = bert_fillmask_sst2(bert_base_tok, bert_base_mlm, sst2_sentences)
report('bert-base-uncased', 'sst2', sst2_gold, pred_base)

print('Evaluating bert-large-uncased on SST-2...')
pred_large = bert_fillmask_sst2(bert_large_tok, bert_large_mlm, sst2_sentences)
report('bert-large-uncased', 'sst2', sst2_gold, pred_large)


Evaluating bert-base-uncased on SST-2...
bert-base-uncased          acc=0.565
Evaluating bert-large-uncased on SST-2...
bert-large-uncased         acc=0.580


0.58

#### GPT-2 — likelihood scoring


In [16]:
print('Loading gpt2...')
gpt2_tok = AutoTokenizer.from_pretrained('gpt2')
gpt2_model = AutoModelForCausalLM.from_pretrained('gpt2').to(device).eval()
print('Done.')


Loading gpt2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Done.


In [17]:
def causal_label_score(tokenizer, model, prompt, label_str):
    """Return log P(label_str tokens | prompt) under a causal LM."""
    full = prompt + label_str
    enc_full = tokenizer(full, return_tensors='pt').to(device)
    enc_prompt = tokenizer(prompt, return_tensors='pt').to(device)
    n_prompt = enc_prompt['input_ids'].shape[1]
    with torch.no_grad():
        out = model(**enc_full, labels=enc_full['input_ids'])
        # recompute per-token log-probs
        logprobs = torch.log_softmax(out.logits[0], dim=-1)
        ids = enc_full['input_ids'][0]
        # sum log-prob only for label tokens
        score = sum(
            logprobs[i - 1, ids[i]].item()
            for i in range(n_prompt, len(ids))
        )
    return score

def gpt2_sst2(tokenizer, model, sentences):
    preds = []
    for sent in sentences:
        prompt = (
            'Review: a waste of time. Sentiment: Negative\n'
            'Review: an absolute masterpiece. Sentiment: Positive\n'
            f'Review: {sent} Sentiment:'
        )
        s_pos = causal_label_score(tokenizer, model, prompt, ' Positive')
        s_neg = causal_label_score(tokenizer, model, prompt, ' Negative')
        preds.append(1 if s_pos > s_neg else 0)
    return preds

print('Evaluating GPT-2 on SST-2...')
pred_gpt2 = gpt2_sst2(gpt2_tok, gpt2_model, sst2_sentences)
report('gpt2', 'sst2', sst2_gold, pred_gpt2)


Evaluating GPT-2 on SST-2...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


gpt2                       acc=0.725


0.725

#### Qwen3 — chat-format zero-shot


In [19]:
print('Loading Qwen/Qwen3-0.6B...')
qwen06_tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B')
qwen06_model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen3-0.6B', torch_dtype=torch.bfloat16
).to(device).eval()

print('Loading Qwen/Qwen3-1.7B...')
qwen17_tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-1.7B')
qwen17_model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen3-1.7B', torch_dtype=torch.bfloat16
).to(device).eval()
print('Done.')


Loading Qwen/Qwen3-0.6B...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading Qwen/Qwen3-1.7B...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Done.


In [20]:
SYSTEM_SST2 = (
    'You are a sentiment classifier. '
    'Reply with exactly one word: Positive or Negative.'
)

def qwen_sst2(tokenizer, model, sentences):
    preds = []
    pos_id = tokenizer.convert_tokens_to_ids('Positive')
    neg_id = tokenizer.convert_tokens_to_ids('Negative')
    for sent in sentences:
        messages = [
            {'role': 'system', 'content': SYSTEM_SST2},
            {'role': 'user',   'content': f'Sentence: {sent}'},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
        inputs = tokenizer(text, return_tensors='pt').to(device)
        with torch.no_grad():
            out = model(**inputs)
        next_logits = out.logits[0, -1, :]   # logits for next token
        p_pos = next_logits[pos_id].item()
        p_neg = next_logits[neg_id].item()
        preds.append(1 if p_pos > p_neg else 0)
    return preds

print('Evaluating Qwen3-0.6B on SST-2...')
pred_q06 = qwen_sst2(qwen06_tok, qwen06_model, sst2_sentences)
report('Qwen3-0.6B', 'sst2', sst2_gold, pred_q06)

print('Evaluating Qwen3-1.7B on SST-2...')
pred_q17 = qwen_sst2(qwen17_tok, qwen17_model, sst2_sentences)
report('Qwen3-1.7B', 'sst2', sst2_gold, pred_q17)


Evaluating Qwen3-0.6B on SST-2...
Qwen3-0.6B                 acc=0.760
Evaluating Qwen3-1.7B on SST-2...
Qwen3-1.7B                 acc=0.880


0.88

### 3.3 MRPC — Paraphrase Detection

**Templates:**
- BERT fill-mask: `"Sentence 1: {s1}\nSentence 2: {s2}\nAre they paraphrases? [MASK]."` → P(`Yes`) vs P(`No`)
- GPT-2 likelihood: compare score of `" Yes"` vs `" No`"
- Qwen3 chat: explicit yes/no question


In [21]:
mrpc_val = mrpc['validation'].select(range(EVAL_N))
mrpc_gold = mrpc_val['label']
mrpc_s1 = mrpc_val['sentence1']
mrpc_s2 = mrpc_val['sentence2']
print(f'MRPC validation sample: {len(mrpc_val)} examples')
print('Label distribution:', dict(zip(*np.unique(mrpc_gold, return_counts=True))))


MRPC validation sample: 200 examples
Label distribution: {np.int64(0): np.int64(63), np.int64(1): np.int64(137)}


In [22]:
def bert_fillmask_pair(tokenizer, model, s1_list, s2_list,
                        pos_word='Yes', neg_word='No',
                        template='Are they paraphrases? [MASK].'):
    pos_id = tokenizer.convert_tokens_to_ids(pos_word)
    neg_id = tokenizer.convert_tokens_to_ids(neg_word)
    preds = []
    for s1, s2 in zip(s1_list, s2_list):
        text = f'Sentence 1: {s1} Sentence 2: {s2} {template}'
        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(device)
        mask_idx = (inputs['input_ids'][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]
        with torch.no_grad():
            logits = model(**inputs).logits[0, mask_idx, :]
        preds.append(1 if logits[0, pos_id] > logits[0, neg_id] else 0)
    return preds

print('BERT-base on MRPC...')
pred_base = bert_fillmask_pair(bert_base_tok, bert_base_mlm, mrpc_s1, mrpc_s2)
report('bert-base-uncased', 'mrpc', mrpc_gold, pred_base)

print('BERT-large on MRPC...')
pred_large = bert_fillmask_pair(bert_large_tok, bert_large_mlm, mrpc_s1, mrpc_s2)
report('bert-large-uncased', 'mrpc', mrpc_gold, pred_large)


BERT-base on MRPC...
bert-base-uncased          acc=0.315  F1=0.000
BERT-large on MRPC...
bert-large-uncased         acc=0.315  F1=0.000


0.315

In [23]:
def gpt2_pair(tokenizer, model, s1_list, s2_list, task_desc, shot_yes, shot_no):
    preds = []
    for s1, s2 in zip(s1_list, s2_list):
        prompt = (
            f'{shot_yes}\n{shot_no}\n'
            f'Sentence 1: {s1}\nSentence 2: {s2}\n{task_desc}'
        )
        s_yes = causal_label_score(tokenizer, model, prompt, ' Yes')
        s_no  = causal_label_score(tokenizer, model, prompt, ' No')
        preds.append(1 if s_yes > s_no else 0)
    return preds

print('GPT-2 on MRPC...')
pred_gpt2 = gpt2_pair(
    gpt2_tok, gpt2_model, mrpc_s1, mrpc_s2,
    task_desc='Are they paraphrases?',
    shot_yes='Sentence 1: The car crashed into the wall. Sentence 2: The vehicle hit the wall. Are they paraphrases? Yes',
    shot_no='Sentence 1: He bought a new car. Sentence 2: She sold her old bike. Are they paraphrases? No',
)
report('gpt2', 'mrpc', mrpc_gold, pred_gpt2)


GPT-2 on MRPC...
gpt2                       acc=0.530  F1=0.608


0.53

In [24]:
SYSTEM_MRPC = (
    'You are a paraphrase detector. '
    'Given two sentences, reply with exactly one word: Yes or No.'
)

def qwen_pair(tokenizer, model, s1_list, s2_list, system_prompt, user_template):
    preds = []
    yes_id = tokenizer.convert_tokens_to_ids('Yes')
    no_id  = tokenizer.convert_tokens_to_ids('No')
    for s1, s2 in zip(s1_list, s2_list):
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_template.format(s1=s1, s2=s2)},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
        inputs = tokenizer(text, return_tensors='pt').to(device)
        with torch.no_grad():
            out = model(**inputs)
        logits = out.logits[0, -1, :]
        preds.append(1 if logits[yes_id] > logits[no_id] else 0)
    return preds

MRPC_TMPL = 'Sentence 1: {s1}\nSentence 2: {s2}\nAre they paraphrases?'

print('Qwen3-0.6B on MRPC...')
pred_q06 = qwen_pair(qwen06_tok, qwen06_model, mrpc_s1, mrpc_s2, SYSTEM_MRPC, MRPC_TMPL)
report('Qwen3-0.6B', 'mrpc', mrpc_gold, pred_q06)

print('Qwen3-1.7B on MRPC...')
pred_q17 = qwen_pair(qwen17_tok, qwen17_model, mrpc_s1, mrpc_s2, SYSTEM_MRPC, MRPC_TMPL)
report('Qwen3-1.7B', 'mrpc', mrpc_gold, pred_q17)


Qwen3-0.6B on MRPC...
Qwen3-0.6B                 acc=0.695  F1=0.818
Qwen3-1.7B on MRPC...
Qwen3-1.7B                 acc=0.745  F1=0.827


0.745

### 3.4 RTE — Recognizing Textual Entailment

Binary label: `0` = entailment, `1` = not entailment.  
Small dataset (only 2.5K training examples) — a classic challenge task.

**Templates:**
- BERT fill-mask: `"Premise: {p}\nHypothesis: {h}\nEntailment? [MASK]."` → P(`Yes`) vs P(`No`)
- GPT-2 likelihood: compare `" Yes"` vs `" No"` after the NLI prompt
- Qwen3 chat: *Given the premise, does the hypothesis follow?*


In [25]:
rte = load_dataset('nyu-mll/glue', 'rte')
rte_val = rte['validation'].select(range(min(EVAL_N, len(rte['validation']))))
rte_gold = rte_val['label']
rte_s1 = rte_val['sentence1']  # premise
rte_s2 = rte_val['sentence2']  # hypothesis
print(f'RTE validation sample: {len(rte_val)} examples')
print('Label distribution:', dict(zip(*np.unique(rte_gold, return_counts=True))))
print('\nExample:')
print('  Premise:   ', rte_val[0]['sentence1'])
print('  Hypothesis:', rte_val[0]['sentence2'])
print('  Label (0=entail, 1=not):', rte_val[0]['label'])


RTE validation sample: 200 examples
Label distribution: {np.int64(0): np.int64(108), np.int64(1): np.int64(92)}

Example:
  Premise:    Dana Reeve, the widow of the actor Christopher Reeve, has died of lung cancer at age 44, according to the Christopher Reeve Foundation.
  Hypothesis: Christopher Reeve had an accident.
  Label (0=entail, 1=not): 1


In [26]:
# BERT fill-mask for RTE
# Note: entailment=0 maps to Yes, not-entailment=1 maps to No
def bert_fillmask_rte(tokenizer, model, premises, hypotheses):
    yes_id = tokenizer.convert_tokens_to_ids('Yes')
    no_id  = tokenizer.convert_tokens_to_ids('No')
    preds = []
    for p, h in zip(premises, hypotheses):
        text = f'Premise: {p} Hypothesis: {h} Entailment? [MASK].'
        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(device)
        mask_idx = (inputs['input_ids'][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]
        with torch.no_grad():
            logits = model(**inputs).logits[0, mask_idx, :]
        preds.append(0 if logits[0, yes_id] > logits[0, no_id] else 1)
    return preds

print('BERT-base on RTE...')
pred_base = bert_fillmask_rte(bert_base_tok, bert_base_mlm, rte_s1, rte_s2)
report('bert-base-uncased', 'rte', rte_gold, pred_base)

print('BERT-large on RTE...')
pred_large = bert_fillmask_rte(bert_large_tok, bert_large_mlm, rte_s1, rte_s2)
report('bert-large-uncased', 'rte', rte_gold, pred_large)


BERT-base on RTE...
bert-base-uncased          acc=0.460
BERT-large on RTE...
bert-large-uncased         acc=0.460


0.46

In [27]:
# GPT-2 on RTE
def gpt2_rte(tokenizer, model, premises, hypotheses):
    preds = []
    for p, h in zip(premises, hypotheses):
        prompt = (
            'Premise: The cat is on the mat. Hypothesis: The cat is near the mat. Entailment? Yes\n'
            'Premise: The sky is blue. Hypothesis: The sky is green. Entailment? No\n'
            f'Premise: {p} Hypothesis: {h} Entailment?'
        )
        s_yes = causal_label_score(tokenizer, model, prompt, ' Yes')
        s_no  = causal_label_score(tokenizer, model, prompt, ' No')
        preds.append(0 if s_yes > s_no else 1)
    return preds

print('GPT-2 on RTE...')
pred_gpt2 = gpt2_rte(gpt2_tok, gpt2_model, rte_s1, rte_s2)
report('gpt2', 'rte', rte_gold, pred_gpt2)


GPT-2 on RTE...
gpt2                       acc=0.460


0.46

In [28]:
SYSTEM_RTE = (
    'You are a textual entailment classifier. '
    'Given a premise and a hypothesis, reply with exactly one word: Yes or No, '
    'where Yes means the hypothesis is entailed by the premise.'
)
RTE_TMPL = 'Premise: {s1}\nHypothesis: {s2}\nDoes the premise entail the hypothesis?'

def qwen_rte(tokenizer, model, premises, hypotheses):
    preds = []
    yes_id = tokenizer.convert_tokens_to_ids('Yes')
    no_id  = tokenizer.convert_tokens_to_ids('No')
    for p, h in zip(premises, hypotheses):
        messages = [
            {'role': 'system', 'content': SYSTEM_RTE},
            {'role': 'user',   'content': RTE_TMPL.format(s1=p, s2=h)},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
        inputs = tokenizer(text, return_tensors='pt').to(device)
        with torch.no_grad():
            out = model(**inputs)
        logits = out.logits[0, -1, :]
        preds.append(0 if logits[yes_id] > logits[no_id] else 1)
    return preds

print('Qwen3-0.6B on RTE...')
pred_q06 = qwen_rte(qwen06_tok, qwen06_model, rte_s1, rte_s2)
report('Qwen3-0.6B', 'rte', rte_gold, pred_q06)

print('Qwen3-1.7B on RTE...')
pred_q17 = qwen_rte(qwen17_tok, qwen17_model, rte_s1, rte_s2)
report('Qwen3-1.7B', 'rte', rte_gold, pred_q17)


Qwen3-0.6B on RTE...
Qwen3-0.6B                 acc=0.690
Qwen3-1.7B on RTE...
Qwen3-1.7B                 acc=0.815


0.815

### 3.5 MNLI — 3-class Natural Language Inference

Label: `0` = entailment, `1` = neutral, `2` = contradiction.  
Harder than RTE because of the three-way distinction and diverse genres.

**Templates:**
- BERT fill-mask: compare P(`Yes`) / P(`Maybe`) / P(`No`) at `[MASK]`
- GPT-2 likelihood: compare scores of `" entailment"`, `" neutral"`, `" contradiction"`
- Qwen3 chat: classify as entailment / neutral / contradiction


In [29]:
mnli_val = mnli['validation_matched'].select(range(EVAL_N))
mnli_gold = mnli_val['label']
mnli_s1 = mnli_val['premise']
mnli_s2 = mnli_val['hypothesis']
print(f'MNLI matched-val sample: {len(mnli_val)} examples')
print('Label distribution:', dict(zip(*np.unique(mnli_gold, return_counts=True))))


MNLI matched-val sample: 200 examples
Label distribution: {np.int64(0): np.int64(82), np.int64(1): np.int64(59), np.int64(2): np.int64(59)}


In [30]:
def bert_fillmask_mnli(tokenizer, model, premises, hypotheses):
    yes_id   = tokenizer.convert_tokens_to_ids('Yes')
    maybe_id = tokenizer.convert_tokens_to_ids('Maybe')
    no_id    = tokenizer.convert_tokens_to_ids('No')
    preds = []
    for p, h in zip(premises, hypotheses):
        text = f'Premise: {p} Hypothesis: {h} True, [MASK], or False?'
        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(device)
        mask_idx = (inputs['input_ids'][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]
        with torch.no_grad():
            logits = model(**inputs).logits[0, mask_idx, :]
        scores = [logits[0, yes_id].item(), logits[0, maybe_id].item(), logits[0, no_id].item()]
        preds.append(int(np.argmax(scores)))  # 0=entail,1=neutral,2=contradict
    return preds

print('BERT-base on MNLI...')
pred_base = bert_fillmask_mnli(bert_base_tok, bert_base_mlm, mnli_s1, mnli_s2)
report('bert-base-uncased', 'mnli', mnli_gold, pred_base)

print('BERT-large on MNLI...')
pred_large = bert_fillmask_mnli(bert_large_tok, bert_large_mlm, mnli_s1, mnli_s2)
report('bert-large-uncased', 'mnli', mnli_gold, pred_large)


BERT-base on MNLI...
bert-base-uncased          acc=0.410  Macro-F1=0.194
BERT-large on MNLI...
bert-large-uncased         acc=0.410  Macro-F1=0.194


0.41

In [31]:
def gpt2_mnli(tokenizer, model, premises, hypotheses):
    preds = []
    for p, h in zip(premises, hypotheses):
        prompt = (
            'Premise: The cat sat on the mat. Hypothesis: The cat is indoors. Relation: entailment\n'
            'Premise: The dog ran in the park. Hypothesis: The dog swam in the lake. Relation: contradiction\n'
            'Premise: Mary went to the store. Hypothesis: Mary bought milk. Relation: neutral\n'
            f'Premise: {p} Hypothesis: {h} Relation:'
        )
        s_e = causal_label_score(tokenizer, model, prompt, ' entailment')
        s_n = causal_label_score(tokenizer, model, prompt, ' neutral')
        s_c = causal_label_score(tokenizer, model, prompt, ' contradiction')
        preds.append(int(np.argmax([s_e, s_n, s_c])))
    return preds

print('GPT-2 on MNLI...')
pred_gpt2 = gpt2_mnli(gpt2_tok, gpt2_model, mnli_s1, mnli_s2)
report('gpt2', 'mnli', mnli_gold, pred_gpt2)


GPT-2 on MNLI...
gpt2                       acc=0.335  Macro-F1=0.267


0.335

In [35]:
SYSTEM_MNLI = (
    'You are a natural language inference classifier. '
    'Given a premise and hypothesis, reply with exactly one word: '
    'entailment, neutral, or contradiction.'
)
MNLI_TMPL = 'Premise: {s1}\nHypothesis: {s2}\nRelation:'

def qwen_mnli(tokenizer, model, premises, hypotheses):
    # Labels are multi-token in Qwen3's BPE vocab, so we score the full
    # label sequence via causal_label_score rather than a single-token logit.
    preds = []
    for p, h in zip(premises, hypotheses):
        messages = [
            {'role': 'system', 'content': SYSTEM_MNLI},
            {'role': 'user',   'content': MNLI_TMPL.format(s1=p, s2=h)},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
        s_e = causal_label_score(tokenizer, model, prompt, ' entailment')
        s_n = causal_label_score(tokenizer, model, prompt, ' neutral')
        s_c = causal_label_score(tokenizer, model, prompt, ' contradiction')
        preds.append(int(np.argmax([s_e, s_n, s_c])))
    return preds

print('Qwen3-0.6B on MNLI...')
pred_q06 = qwen_mnli(qwen06_tok, qwen06_model, mnli_s1, mnli_s2)
report('Qwen3-0.6B', 'mnli', mnli_gold, pred_q06)

print('Qwen3-1.7B on MNLI...')
pred_q17 = qwen_mnli(qwen17_tok, qwen17_model, mnli_s1, mnli_s2)
report('Qwen3-1.7B', 'mnli', mnli_gold, pred_q17)

Qwen3-0.6B on MNLI...
Qwen3-0.6B                 acc=0.455  Macro-F1=0.288
Qwen3-1.7B on MNLI...
Qwen3-1.7B                 acc=0.775  Macro-F1=0.771


0.775